# ChiNeutro: evaluación de Qwen3.5-2B en Reddit

Este notebook **no entrena** el modelo. Carga el adaptador LoRA guardado por `entrenar_qwen35.ipynb`, genera predicciones para `frases` del JSON de Reddit y calcula métricas que pueden obtenerse sin referencias de neutralización.

**Se mantiene la nomenclatura del notebook de entrenamiento:**
- `MODELO_ID = "Qwen/Qwen3.5-2B"`
- `MODO = "lora"`
- `LORA_R = 16`
- `NOMBRE = "Qwen3_5_2B_lora_r16"`
- adaptador: `salidas_qwen/Qwen3_5_2B_lora_r16/adaptador_final`

El archivo de Reddit no venía adjunto, así que debes indicar su ruta en `RUTA_REDDIT` en la celda de configuración.

**Importante sobre las métricas:** el JSON de Reddit mostrado no contiene una frase neutralizada de referencia. Por eso BLEU queda en `null`. BERTScore se calcula contra la entrada (`bertscore_contra = "entrada"`), y el porcentaje de términos retenidos se calcula usando `expresiones[].forma`/`clave`. La evaluación con juez queda pendiente y se refleja en `juez_faltan`; este notebook no inventa resultados del juez.

In [3]:
# 1. Variables de entorno y paquetes
import os
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import importlib.metadata as md

paquetes = ["torch", "transformers", "peft", "accelerate", "pandas", "bert-score"]
faltan = []
for p in paquetes:
    try:
        print(f"{p:14s} {md.version(p)}")
    except md.PackageNotFoundError:
        print(f"{p:14s} NO instalado")
        faltan.append(p)

if faltan:
    raise RuntimeError(
        "Faltan paquetes. Ejecuta en una celda: %pip install -U "
        + " ".join(faltan)
        + " ; luego reinicia el kernel."
    )


torch          2.14.0+cu130
transformers   5.18.0
peft           0.22.0
accelerate     1.15.0
pandas         3.0.5
bert-score     0.3.13


In [5]:
# 2. Configuración: ajusta solamente RUTA_REDDIT si es necesario
from pathlib import Path
import re, json, sys, gc, unicodedata
import torch

MODELO_ID = "Qwen/Qwen3.5-2B"
MODO = "lora"
LORA_R, LORA_ALPHA = 16, 32
MAX_NEW_TOKENS = 96
BATCH_SIZE = 8
SEED = 42

corto = MODELO_ID.split("/")[-1].replace("-", "_")
NOMBRE = f"{corto}_{MODO}_r{LORA_R}"
print("Corrida:", NOMBRE)

# Deja este notebook dentro del repositorio NLP-ChiNeutro.
marca = "data/entrenamiento/splits/split_oficial.csv"
actual = Path.cwd().resolve()
RAIZ = next((d for d in [actual, *actual.parents] if (d / marca).is_file()), None)
assert RAIZ is not None, (
    f"No encontré {marca} subiendo desde {actual}. "
    "Abre este notebook desde una carpeta dentro de tu repo."
)

DIR_ADAPTADOR = (
    RAIZ
    / "models"
    / "Qwen3.5"
    / "2B"
    / "salidas_qwen"
    / NOMBRE
    / "adaptador_final"
)

# CAMBIA esta ruta por la ubicación real del JSON de Reddit en tu repositorio.
RUTA_REDDIT = RAIZ / "data" / "reddit" / "reddit.json"

# Salidas siguiendo la organización de predicciones de Qwen.
DIR_PRED = RAIZ / "models" / "Qwen3.5" / "2B" / "predicciones" / "reddit"
DIR_PRED.mkdir(parents=True, exist_ok=True)

print("Repo:", RAIZ)
print("Adaptador:", DIR_ADAPTADOR)
print("Dataset Reddit:", RUTA_REDDIT)
print("Salidas:", DIR_PRED)
assert RUTA_REDDIT.is_file(), (
    f"No existe el JSON de Reddit: {RUTA_REDDIT}\n"
    "Corrige RUTA_REDDIT en esta celda."
)

assert torch.cuda.is_available(), "No se detectó CUDA. Este notebook está preparado para inferencia local con GPU."
BF16 = torch.cuda.get_device_capability(0)[0] >= 8
DTYPE = torch.bfloat16 if BF16 else torch.float16
print("GPU:", torch.cuda.get_device_name(0), "| dtype:", DTYPE)


Corrida: Qwen3.5_2B_lora_r16
Repo: C:\Users\conso\OneDrive\Desktop\Universidah\2026-2\NLP\Proyecto\NLP-ChiNeutro
Adaptador: C:\Users\conso\OneDrive\Desktop\Universidah\2026-2\NLP\Proyecto\NLP-ChiNeutro\models\Qwen3.5\2B\salidas_qwen\Qwen3.5_2B_lora_r16\adaptador_final
Dataset Reddit: C:\Users\conso\OneDrive\Desktop\Universidah\2026-2\NLP\Proyecto\NLP-ChiNeutro\data\reddit\reddit.json
Salidas: C:\Users\conso\OneDrive\Desktop\Universidah\2026-2\NLP\Proyecto\NLP-ChiNeutro\models\Qwen3.5\2B\predicciones\reddit
GPU: NVIDIA GeForce RTX 5070 | dtype: torch.bfloat16


In [6]:
# 3. Leer Reddit y revisar su estructura
with open(RUTA_REDDIT, "r", encoding="utf-8") as f:
    data_reddit = json.load(f)

assert "frases" in data_reddit, "El JSON no contiene la clave 'frases'."
frases_reddit = data_reddit["frases"]
assert isinstance(frases_reddit, list) and frases_reddit, "La lista 'frases' está vacía o no es válida."

for i, item in enumerate(frases_reddit):
    assert "frase" in item, f"El registro {i} no contiene 'frase'."

print("Metadatos:", data_reddit.get("meta", {}))
print("Cantidad de frases:", len(frases_reddit))
print("\nEjemplo:")
print(json.dumps(frases_reddit[0], ensure_ascii=False, indent=2)[:2500])


Metadatos: {'fuente': 'Reddit (r/chile, r/RepublicadeChile) vía API de Arctic Shift, muestreo aleatorio por fecha', 'fecha_extraccion': '2026-11-19', 'comentarios_leidos': 143074, 'oraciones_candidatas': 20379, 'expresiones_cubiertas': 518, 'parametros': {'cmd': None, 'objetivo': 2000, 'tope': 5, 'min_palabras': 5, 'max_palabras': 40, 'semilla': 0}}
Cantidad de frases: 1900

Ejemplo:
{
  "id": "reddit_00000",
  "comentario_id": "k55ndnt",
  "subreddit": "chile",
  "fecha": "2023-10-16",
  "score": 23,
  "frase": "como te poni a llorar por un chocmans",
  "expresiones": [
    {
      "clave": "como te poni a llorar por un chocmans",
      "termino": "¿Cómo te poní a llorar por un chocmans?",
      "forma": "como te poni a llorar por un chocmans"
    }
  ],
  "contexto": "como te poni a llorar por un chocmans"
}


In [7]:
# 4. Cargar el mismo prompt utilizado durante el entrenamiento
# Importamos INSTRUCCION_CHAT desde el módulo original de Gemma,
# tal como lo hace entrenar_qwen35.ipynb.
sys.path.insert(0, str(RAIZ))
sys.path.insert(0, str(RAIZ / "models" / "gemma4"))
from comun import INSTRUCCION_CHAT

def entrada(frase):
    return INSTRUCCION_CHAT.format(chilena=frase.strip())

from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(DIR_ADAPTADOR)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

FIN_TURNO = "<|im_end|>"
ID_FIN = tok.convert_tokens_to_ids(FIN_TURNO)
assert ID_FIN is not None, f"El tokenizador no tiene {FIN_TURNO}"

def prompt_de(frase, pensar=False):
    mensajes = [{"role": "user", "content": entrada(frase)}]
    return tok.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=pensar
    )

print(prompt_de(frases_reddit[0]["frase"]))


<|im_start|>user
Traduce la siguiente frase del español de Chile al español neutro. Responde solo con la traducción.

como te poni a llorar por un chocmans<|im_end|>
<|im_start|>assistant
<think>

</think>




In [8]:
# 5. Cargar el modelo base y el adaptador LoRA entrenado
from transformers import AutoModelForCausalLM, set_seed
from peft import PeftModel

set_seed(SEED)
modelo_base = AutoModelForCausalLM.from_pretrained(
    MODELO_ID,
    dtype=DTYPE,
    device_map={"": 0}
)
modelo = PeftModel.from_pretrained(modelo_base, DIR_ADAPTADOR)
modelo.eval()
modelo.config.use_cache = True

tok.padding_side = "left"
print(type(modelo).__name__)
print(f"Memoria GPU asignada: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

W1124 19:21:52.677000 17536 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


PeftModelForCausalLM
Memoria GPU asignada: 3.83 GB


In [9]:
# 6. Función de generación: misma estrategia que entrenar_qwen35.ipynb
@torch.no_grad()
def traducir(frases, bs=BATCH_SIZE):
    modelo.eval()
    salidas = []
    for i in range(0, len(frases), bs):
        lote = frases[i:i + bs]
        x = tok(
            [prompt_de(f) for f in lote],
            return_tensors="pt",
            padding=True,
            add_special_tokens=False
        ).to(modelo.device)

        out = modelo.generate(
            **x,
            do_sample=False,
            num_beams=1,
            temperature=None,
            top_p=None,
            top_k=None,
            max_new_tokens=MAX_NEW_TOKENS,
            use_cache=True,
            eos_token_id=list({ID_FIN, tok.eos_token_id} - {None}),
            pad_token_id=tok.pad_token_id
        )

        textos = tok.batch_decode(
            out[:, x["input_ids"].shape[1]:],
            skip_special_tokens=True
        )
        for s in textos:
            s = re.sub(r"<think>.*?</think>", "", s, flags=re.S)
            salidas.append(s.strip().split("\n")[0].strip())
    return salidas

# Prueba pequeña antes de procesar todo Reddit.
muestra = [x["frase"] for x in frases_reddit[:min(3, len(frases_reddit))]]
for original, pred in zip(muestra, traducir(muestra, bs=min(3, len(muestra)))):
    print("ENTRADA:   ", original)
    print("PREDICCIÓN:", pred)
    print("---")


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


ENTRADA:    como te poni a llorar por un chocmans
PREDICCIÓN: como te pones a llorar por una tontería
---
ENTRADA:    Creo que el dicho "Muere de viejo y no de sapo" le viene perfecto a esto
PREDICCIÓN: Creo que la frase "Muere de viejo y no de tonto" le viene perfecto a esto
---
ENTRADA:    Vinieron los de siempre, comimos y se me echó la yegua, ni siquiera comí tanto, estoy aguantando cada vez menos (igual mucho mejor que el 2017, estaba sin pega y lo pasé totalmente sólo, sólo).
PREDICCIÓN: Vinieron los de siempre, comimos y me desanimé, ni siquiera comí tanto, estoy aguantando cada vez menos (aunque mucho mejor que en 2017, estaba sin trabajo y lo pasé totalmente bien).
---


In [10]:
# 7. Generar todas las predicciones de Reddit
entradas_reddit = [x["frase"].strip() for x in frases_reddit]
predicciones = traducir(entradas_reddit)

assert len(predicciones) == len(frases_reddit), "La cantidad de predicciones no coincide con Reddit."

# Se conserva el esquema de columnas que usa evaluacion/ en el proyecto.
# id_mistral se reutiliza como columna identificadora compatible con ese formato;
# su contenido es el id de Reddit (reddit_XXXXX), no un id de Mistral.
import pandas as pd

filas = []
for item, pred in zip(frases_reddit, predicciones):
    filas.append({
        "id_mistral": item.get("id", ""),
        "frase_chilena": item["frase"],
        "prediccion": pred
    })

df_predicciones = pd.DataFrame(filas, columns=["id_mistral", "frase_chilena", "prediccion"])
ARCHIVO_CSV = DIR_PRED / f"predicciones_{NOMBRE}_reddit.csv"
df_predicciones.to_csv(ARCHIVO_CSV, index=False, encoding="utf-8")

# Archivo JSONL auxiliar: mantiene contexto y metadatos originales por registro.
ARCHIVO_DETALLE = DIR_PRED / f"predicciones_{NOMBRE}_reddit_detalle.jsonl"
with open(ARCHIVO_DETALLE, "w", encoding="utf-8") as f:
    for item, pred in zip(frases_reddit, predicciones):
        registro = dict(item)
        registro["prediccion"] = pred
        f.write(json.dumps(registro, ensure_ascii=False) + "\n")

print("CSV:", ARCHIVO_CSV)
print("Detalle:", ARCHIVO_DETALLE)
print("Predicciones:", len(df_predicciones))
print("Vacías:", int((df_predicciones["prediccion"].str.strip() == "").sum()))
print("Copia literal:", (
    df_predicciones["prediccion"].str.strip().str.lower()
    == df_predicciones["frase_chilena"].str.strip().str.lower()
).mean())
display(df_predicciones.head(10))


CSV: C:\Users\conso\OneDrive\Desktop\Universidah\2026-2\NLP\Proyecto\NLP-ChiNeutro\models\Qwen3.5\2B\predicciones\reddit\predicciones_Qwen3.5_2B_lora_r16_reddit.csv
Detalle: C:\Users\conso\OneDrive\Desktop\Universidah\2026-2\NLP\Proyecto\NLP-ChiNeutro\models\Qwen3.5\2B\predicciones\reddit\predicciones_Qwen3.5_2B_lora_r16_reddit_detalle.jsonl
Predicciones: 1900
Vacías: 0
Copia literal: 0.02526315789473684


,id_mistral,frase_chilena,prediccion
0,reddit_00000,como te poni a llorar por un chocmans,como te pones a llorar por una tontería
1,reddit_00001,"Creo que el dicho ""Muere de viejo y no de sapo...","Creo que la frase ""Muere de viejo y no de tont..."
2,reddit_00002,"Vinieron los de siempre, comimos y se me echó ...","Vinieron los de siempre, comimos y me desanimé..."
3,reddit_00003,Esa wea de cenar a la hora del pico y pa navid...,Esa cosa de cenar a la hora del desastre y par...
4,reddit_00004,Anda a lavarte la raja Third World Trash,Anda a lavarte la cara Third World Trash
5,reddit_00005,"(se me cayo el carnet, probablemente los ramos...","(se me cae la edad, probablemente los ramos ah..."
6,reddit_00006,Capucha: Te paso por el pico vendío y la conch...,"Carajo: Te paso por el culo, idiota y la mierda"
7,reddit_00007,otra cosa es con guitarra.,Otra cosa es con la guitarra.
8,reddit_00008,Andai con la pera po papito.,Andas con la pata para papá.
9,reddit_00009,Sabi que la opinión va a estar de la perra cua...,Sabes que la opinión va a estar excelente cuan...


## 8. Métricas automáticas disponibles para Reddit

- **BLEU:** queda `null` porque el JSON no proporciona traducciones/neutralizaciones de referencia.
- **BERTScore F1:** se calcula entre la predicción y la frase de entrada, siguiendo `bertscore_contra = "entrada"`. Es una medida de similitud con la entrada, no una medida directa de calidad de neutralización.
- **Términos retenidos:** porcentaje de expresiones anotadas en `expresiones` cuya forma (`forma`, o en su defecto `clave`) sigue apareciendo en la predicción. Es una operacionalización sobre las anotaciones de Reddit; conviene confirmar que coincide con la definición exacta del evaluador del grupo.

La evaluación del juez no se ejecuta en esta celda. Se dejan sus campos pendientes para no inventar resultados.

In [11]:
# 9. Calcular términos retenidos y BERTScore
def normalizar(texto):
    texto = str(texto or "").casefold().strip()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    texto = re.sub(r"[^a-z0-9ñ]+", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()

total_expresiones = 0
expresiones_retenidas = 0

for item, pred in zip(frases_reddit, predicciones):
    pred_norm = normalizar(pred)
    for exp in item.get("expresiones", []) or []:
        termino = exp.get("forma") or exp.get("clave") or exp.get("termino")
        if not termino:
            continue
        total_expresiones += 1
        termino_norm = normalizar(termino)
        if termino_norm and re.search(
            r"(?<!\w)" + re.escape(termino_norm) + r"(?!\w)",
            pred_norm
        ):
            expresiones_retenidas += 1

terminos_retenidos = (
    100.0 * expresiones_retenidas / total_expresiones
    if total_expresiones else None
)
print("Expresiones anotadas:", total_expresiones)
print("Expresiones retenidas:", expresiones_retenidas)
print("Términos retenidos (%):", terminos_retenidos)

from bert_score import score as bertscore_score

P, R, F1 = bertscore_score(
    predicciones,
    entradas_reddit,
    lang="es",
    verbose=True,
    batch_size=16
)
bertscore_f1 = float(F1.mean().item())
print("BERTScore F1 contra entrada:", bertscore_f1)


Expresiones anotadas: 2424
Expresiones retenidas: 319
Términos retenidos (%): 13.16006600660066


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

... (4 outputs left out)

  0%|          | 0/235 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/119 [00:00<?, ?it/s]

done in 3.45 seconds, 551.28 sentences/sec
BERTScore F1 contra entrada: 0.8959957957267761


In [12]:
# 10. Guardar resumen JSON con el esquema de evaluación de Gemma
# El juez queda explícitamente pendiente: este notebook no lo ha ejecutado.
n = len(frases_reddit)

resumen = {
    "n": n,
    "bleu": None,
    "terminos_retenidos_%": round(terminos_retenidos, 4) if terminos_retenidos is not None else None,
    "bertscore_f1": round(bertscore_f1, 4),
    "bertscore_contra": "entrada",
    "juez_modelo": None,
    "juez_effort": None,
    "juez_rubrica": None,
    "juez_n": 0,
    "juez_faltan": n,
    "juez_sentido_media": None,
    "juez_fluidez_media": None,
    "juez_jerga_eliminada_%": None,
    "juez_aceptable_%": None
}

ARCHIVO_EVALUACION = DIR_PRED / f"predicciones_{NOMBRE}_reddit_evaluacion.json"
with open(ARCHIVO_EVALUACION, "w", encoding="utf-8") as f:
    json.dump(resumen, f, ensure_ascii=False, indent=2)

print(json.dumps(resumen, ensure_ascii=False, indent=2))
print("\nGuardado:", ARCHIVO_EVALUACION)


{
  "n": 1900,
  "bleu": null,
  "terminos_retenidos_%": 13.1601,
  "bertscore_f1": 0.896,
  "bertscore_contra": "entrada",
  "juez_modelo": null,
  "juez_effort": null,
  "juez_rubrica": null,
  "juez_n": 0,
  "juez_faltan": 1900,
  "juez_sentido_media": null,
  "juez_fluidez_media": null,
  "juez_jerga_eliminada_%": null,
  "juez_aceptable_%": null
}

Guardado: C:\Users\conso\OneDrive\Desktop\Universidah\2026-2\NLP\Proyecto\NLP-ChiNeutro\models\Qwen3.5\2B\predicciones\reddit\predicciones_Qwen3.5_2B_lora_r16_reddit_evaluacion.json


## 11. Qué queda pendiente: evaluación con juez

El JSON generado deja `juez_n = 0` y `juez_faltan = n` porque el notebook de entrenamiento indica que el juez (Opus vía Claude Code) se ejecuta después en el computador, pero no incluye el comando ni la interfaz de ese evaluador.

Cuando tengas el comando/script del juez que utilizó el proyecto para Gemma, se puede agregar aquí y actualizar los campos `juez_*` con resultados reales. No ejecutes el juez sobre un CSV distinto al generado en la sección 7: usa `ARCHIVO_CSV` como entrada.

**Archivos generados:**
- `predicciones_Qwen3_5_2B_lora_r16_reddit.csv`
- `predicciones_Qwen3_5_2B_lora_r16_reddit_detalle.jsonl`
- `predicciones_Qwen3_5_2B_lora_r16_reddit_evaluacion.json`
